# Corpus Exploration & Similarity Analysis

Interactive notebook for exploring the corpus, analyzing similarity score distributions, and visualizing retrieval results.

Run the pipeline (at least through the retrieval step) before using this notebook.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from autoconcourse.config import load_config
from autoconcourse.utils import load_embeddings
from autoconcourse.retrieval import cosine_similarity_matrix, aggregate_anchor_embeddings

## 1. Load Data

In [ ]:
CONFIG_PATH = PROJECT_ROOT / "configs" / "healthcare_ch.yaml"  # <-- Change for your topic
config = load_config(CONFIG_PATH)

cache = config.cache_dir
paragraphs = pd.read_csv(cache / "paragraphs.csv")
print(f"Corpus: {len(paragraphs)} paragraphs")
print(f"Columns: {list(paragraphs.columns)}")
paragraphs.head()

In [ ]:
# Load embeddings
para_emb = load_embeddings(cache / "paragraph_embeddings.npy")
anchor_emb = load_embeddings(cache / "anchor_embeddings.npy")
anchors = pd.read_csv(cache / "anchors.csv")

print(f"Paragraph embeddings: {para_emb.shape}")
print(f"Anchor embeddings: {anchor_emb.shape}")
print(f"Anchors: {len(anchors)} entries across {anchors['category'].nunique()} categories")

## 2. Corpus Statistics

In [ ]:
paragraphs["text_length"] = paragraphs["text"].str.len()
paragraphs["word_count"] = paragraphs["text"].str.split().str.len()

print(f"Text length: mean={paragraphs['text_length'].mean():.0f}, median={paragraphs['text_length'].median():.0f}")
print(f"Word count: mean={paragraphs['word_count'].mean():.0f}, median={paragraphs['word_count'].median():.0f}")

if "language" in paragraphs.columns:
    print(f"\nLanguage distribution:")
    print(paragraphs["language"].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(paragraphs["text_length"], bins=50, edgecolor="black", alpha=0.7)
axes[0].set_xlabel("Text length (characters)")
axes[0].set_ylabel("Count")
axes[0].set_title("Paragraph length distribution")

axes[1].hist(paragraphs["word_count"], bins=50, edgecolor="black", alpha=0.7)
axes[1].set_xlabel("Word count")
axes[1].set_ylabel("Count")
axes[1].set_title("Paragraph word count distribution")

plt.tight_layout()
plt.show()

## 3. Similarity Score Analysis

Compute cosine similarity between all paragraphs and each aggregated anchor category.

In [ ]:
# Aggregate anchors per category and compute similarities
agg_anchors = aggregate_anchor_embeddings(anchor_emb, anchors["category"].values)
cat_ids = sorted(agg_anchors.keys())

subcategories_path = config.resolve_path(config.anchors.subcategories_file)
with open(subcategories_path) as f:
    subcategories = json.load(f)
cat_names = {sc["id"]: sc["name"] for sc in subcategories}

# Compute similarity matrix: (n_paragraphs, n_categories)
anchor_matrix = np.stack([agg_anchors[c] for c in cat_ids])
sim_matrix = cosine_similarity_matrix(para_emb, anchor_matrix)

print(f"Similarity matrix shape: {sim_matrix.shape}")
print(f"Global mean similarity: {sim_matrix.mean():.4f}")
print(f"Global max similarity: {sim_matrix.max():.4f}")

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, cat_id in enumerate(cat_ids[:8]):
    ax = axes[i] if i < len(axes) else None
    if ax is None:
        break
    scores = sim_matrix[:, i]
    ax.hist(scores, bins=50, edgecolor="black", alpha=0.7)
    ax.set_title(f"{cat_names.get(cat_id, cat_id)}", fontsize=9)
    ax.set_xlabel("Cosine similarity")
    ax.axvline(np.percentile(scores, 95), color="red", linestyle="--", alpha=0.7)

plt.suptitle("Similarity score distributions per subcategory (red = 95th percentile)", fontsize=12)
plt.tight_layout()
plt.show()

## 4. Top Retrieved Paragraphs

Inspect the top-k retrieved paragraphs per subcategory.

In [ ]:
TOP_K = 5  # Show top 5 for inspection

for i, cat_id in enumerate(cat_ids):
    scores = sim_matrix[:, i]
    top_idx = np.argsort(scores)[::-1][:TOP_K]
    
    print(f"\n{'='*80}")
    print(f"Subcategory {cat_id}: {cat_names.get(cat_id, '?')}")
    print(f"{'='*80}")
    for rank, idx in enumerate(top_idx, 1):
        text = paragraphs.iloc[idx]["text"]
        print(f"\n  [{rank}] sim={scores[idx]:.4f}")
        print(f"      {text[:200]}...")